In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv("../.env")
engine = create_engine(os.getenv("DATABASE_URL"))

def sql_query(query):
    return pd.read_sql(query, engine)

In [2]:
carga_promociones = """
    SELECT pos."Fecha", c.id_campus, v.id_vertical
    FROM (
        SELECT "Fecha", "Campus", "Vertical" FROM stg_alumnos
        UNION
        SELECT "Fecha", "Campus", "Vertical" FROM stg_claustro
    ) AS pos
    JOIN campus c     ON c.nombre_campus   = pos."Campus"
    JOIN verticales v ON v.nombre_vertical = pos."Vertical"
"""

carga_proyectos = """
    SELECT proy.nombre_proyecto, v.id_vertical
    FROM (
        SELECT DISTINCT REPLACE(REPLACE("Proyecto", 'Proyecto_', ''), 'FullSatck', 'FullStack') AS nombre_proyecto, "Vertical"
        FROM stg_resultados
    ) AS proy
    JOIN verticales v ON v.nombre_vertical = proy."Vertical"
"""

carga_matriculas = """
    SELECT f.id_alumno, p.id_promocion
    FROM (
        SELECT a.id_alumno, "Fecha" AS fecha, "Campus" AS campus, "Vertical" AS vertical
        FROM stg_alumnos s
        JOIN alumnos a ON s."Email" = a.email
    ) AS f
    JOIN campus c      ON f.campus = c.nombre_campus
    JOIN verticales v  ON f.vertical = v.nombre_vertical
    JOIN promociones p ON p.fecha_comienzo = f.fecha
                      AND p.id_campus = c.id_campus
                      AND p.id_vertical = v.id_vertical
"""

carga_profesor_promocion = """
    SELECT f.id_profesor, pr.id_promocion, m.id_modalidad, f.rol
    FROM (
        SELECT p.id_profesor, "Rol" AS rol, "Modalidad" AS modalidad,
               "Fecha" AS fecha, "Campus" AS campus, "Vertical" AS vertical
        FROM stg_claustro sc
        JOIN profesores p ON sc."Nombre" = p.nombre_profesor
    ) AS f
    JOIN campus c       ON f.campus = c.nombre_campus
    JOIN verticales v   ON f.vertical = v.nombre_vertical
    JOIN promociones pr ON pr.fecha_comienzo = f.fecha
                       AND pr.id_campus = c.id_campus
                       AND pr.id_vertical = v.id_vertical
    JOIN modalidad m    ON f.modalidad = m.nombre_modalidad
"""

carga_alumno_proyecto = """
    SELECT m.id_matricula, pj.id_proyecto, f.resultado
    FROM (
        SELECT a.id_alumno,
               REPLACE(REPLACE(s."Proyecto", 'Proyecto_', ''), 'FullSatck', 'FullStack') AS nombre_proyecto,
               s."Resultado" AS resultado, s."Fecha" AS fecha,
               s."Campus" AS campus, s."Vertical" AS vertical
        FROM stg_resultados s
        JOIN alumnos a ON s."Email" = a.email
    ) AS f
    JOIN campus c      ON f.campus = c.nombre_campus
    JOIN verticales v  ON f.vertical = v.nombre_vertical
    JOIN promociones p ON p.fecha_comienzo = f.fecha
                      AND p.id_campus = c.id_campus
                      AND p.id_vertical = v.id_vertical
    JOIN matriculas m  ON m.id_alumno = f.id_alumno
                      AND m.id_promocion = p.id_promocion
    JOIN proyectos pj  ON pj.nombre_proyecto = f.nombre_proyecto
                      AND pj.id_vertical = v.id_vertical
"""

In [3]:
# Para cada tabla: nuestra consulta, y lo que hay en Render
comprobaciones = {
    "promociones": (carga_promociones, "SELECT fecha_comienzo, id_campus, id_vertical FROM promociones"),
    "proyectos": (carga_proyectos, "SELECT nombre_proyecto, id_vertical FROM proyectos"),
    "matriculas": (carga_matriculas, "SELECT id_alumno, id_promocion FROM matriculas"),
    "profesor_promocion": (carga_profesor_promocion, "SELECT id_profesor, id_promocion, id_modalidad, rol FROM profesor_promocion"),
    "alumno_proyecto": (carga_alumno_proyecto, "SELECT id_matricula, id_proyecto, resultado FROM alumno_proyecto"),
}

filas = []
for tabla, (carga, en_render) in comprobaciones.items():
    filas.append({
        "tabla": tabla,
        "filas_carga": len(sql_query(carga)),
        "faltan_en_render": len(sql_query(carga + " EXCEPT " + en_render)),
        "sobran_en_render": len(sql_query(en_render + " EXCEPT " + carga)),
    })

pd.DataFrame(filas)

,tabla,filas_carga,faltan_en_render,sobran_en_render
0,promociones,6,0,0
1,proyectos,10,0,0
2,matriculas,52,0,0
3,profesor_promocion,10,0,0
4,alumno_proyecto,260,0,0


1. Cada promoción tiene los alumnos que le tocan?

In [ ]:
sql_query("""
    SELECT p.fecha_comienzo, c.nombre_campus, v.nombre_vertical, COUNT(m.id_matricula) AS alumnos
    FROM promociones p
    JOIN campus c     
    ON c.id_campus = p.id_campus
    JOIN verticales v 
    ON v.id_vertical = p.id_vertical
    LEFT JOIN matriculas m 
    ON m.id_promocion = p.id_promocion
    GROUP BY p.fecha_comienzo, c.nombre_campus, v.nombre_vertical
    ORDER BY p.fecha_comienzo, c.nombre_campus, v.nombre_vertical
""")

,fecha_comienzo,nombre_campus,nombre_vertical,alumnos
0,2023-09-18,Madrid,Data Science,15
1,2023-09-18,Madrid,Full Stack,14
2,2023-09-18,Valencia,Full Stack,0
3,2024-02-12,Madrid,Data Science,10
4,2024-02-12,Madrid,Full Stack,0
5,2024-02-12,Valencia,Full Stack,13


In [8]:
for archivo in ["clase_1.csv", "clase_2.csv", "clase_3.csv", "clase_4.csv"]:
    df = pd.read_csv(f"../data/{archivo}", sep=";")
    print(archivo, len(df))

clase_1.csv 15
clase_2.csv 10
clase_3.csv 14
clase_4.csv 13


In [5]:
# Resultados de un alumno: comparar a mano con clase_1.csv
sql_query("""
    SELECT a.nombre_alumno, pj.nombre_proyecto, ap.resultado
    FROM alumno_proyecto ap
    JOIN matriculas m ON m.id_matricula = ap.id_matricula
    JOIN alumnos a    ON a.id_alumno    = m.id_alumno
    JOIN proyectos pj ON pj.id_proyecto = ap.id_proyecto
    WHERE a.nombre_alumno = 'Jafet Casals'
""")

,nombre_alumno,nombre_proyecto,resultado
0,Jafet Casals,HLF,Apto
1,Jafet Casals,EDA,No Apto
2,Jafet Casals,BBDD,Apto
3,Jafet Casals,ML,Apto
4,Jafet Casals,Deployment,Apto


In [7]:
clase_1 = pd.read_csv("../data/clase_1.csv", sep=";")

proyectos_ds = [col for col in clase_1.columns if col.startswith("Proyecto_")]

clase_1.loc[clase_1["Nombre"] == "Jafet Casals", ["Nombre"] + proyectos_ds]

,Nombre,Proyecto_HLF,Proyecto_EDA,Proyecto_BBDD,Proyecto_ML,Proyecto_Deployment
0,Jafet Casals,Apto,No Apto,Apto,Apto,Apto
